# Notebook 05 — OPERA-DISP and Sentinel-1 Subsidence Analysis
**Stage 7 of the Ghana Groundwater–Subsidence Research Pipeline**

Documents OPERA-DISP coverage for Ghana, inventories Sentinel-1 SLC scenes, outlines the InSAR processing workflow (SNAP → MintPy), describes the ASF HyP3 on-demand processing option, and generates a preliminary proxy subsidence potential map from available spatial indicators for interim use in the risk index.

In [ ]:
import os
import pathlib; os.chdir(str(pathlib.Path.cwd().parent) if pathlib.Path.cwd().name == 'notebooks' else str(pathlib.Path.cwd()))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
import asf_search as asf
import earthaccess

ROOT      = Path('.')
OUT_SUB   = ROOT / 'data' / 'processed' / 'subsidence'
OUT_INV   = ROOT / 'data' / 'raw' / 'sentinel1'
FIG_DIR   = ROOT / 'outputs' / 'figures'
TAB_DIR   = ROOT / 'outputs' / 'tables'
GW_DIR    = ROOT / 'data' / 'processed' / 'groundwater_anomalies'

for d in [OUT_SUB, OUT_INV, FIG_DIR, TAB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

BBOX     = (-3.5, 4.0, 1.5, 11.5)
WKT_BBOX = 'POLYGON((-3.5 4.0, 1.5 4.0, 1.5 11.5, -3.5 11.5, -3.5 4.0))'
print('Setup OK')


In [ ]:
# 2. OPERA-DISP coverage check
earthaccess.login(strategy='netrc')

opera_results = earthaccess.search_data(
    short_name='OPERA_L3_DISP-S1_V1',
    bounding_box=BBOX,
    temporal=('2016-01-01', '2024-12-31'),
    count=50
)
print(f'OPERA-DISP granules covering coastal Ghana: {len(opera_results)}')

if len(opera_results) == 0:
    print()
    print('NOTE: OPERA-DISP does not cover West Africa / Ghana.')
    print('      Coverage is focused on North America and selected global areas.')
    print('      InSAR analysis requires direct S1 processing (SNAP/MintPy)')
    print('      or ASF HyP3 on-demand service (https://hyp3.asf.alaska.edu).')
    print()
    print('This notebook:')
    print('  (a) Inventories S1 scenes available for Ghana')
    print('  (b) Identifies recommended InSAR pairs for Accra hotspot')
    print('  (c) Generates HyP3 submission script')
    print('  (d) Builds proxy subsidence potential from GWSA+drought+elevation')


In [ ]:
# 3. Sentinel-1 IW SLC scene inventory 2015-2024
print('Querying ASF Sentinel-1 IW SLC scenes ...')

all_scenes = []
for year in range(2015, 2025):
    results = asf.search(
        platform=asf.PLATFORM.SENTINEL1,
        processingLevel='SLC',
        beamMode='IW',
        intersectsWith=WKT_BBOX,
        start=f'{year}-01-01',
        end=f'{year}-12-31',
        maxResults=500
    )
    for r in results:
        p = r.properties
        all_scenes.append({
            'scene_name':   p.get('sceneName', ''),
            'date':         p.get('startTime', '')[:10],
            'year':         year,
            'path':         p.get('pathNumber'),
            'frame':        p.get('frameNumber'),
            'platform':     p.get('platform', ''),
            'orbit_dir':    p.get('flightDirection', ''),
            'polarisation': p.get('polarization', ''),
            'file_size_mb': round(p.get('bytes', 0) / 1e6, 1),
        })
    print(f'  {year}: {len(results)} scenes')

df_s1 = pd.DataFrame(all_scenes)
df_s1['date'] = pd.to_datetime(df_s1['date'])
df_s1 = df_s1.sort_values('date').reset_index(drop=True)
df_s1.to_csv(OUT_INV / 's1_scene_inventory_ghana_2015_2024.csv', index=False)

print(f'Total scenes: {len(df_s1)}')
print(f'Paths: {sorted(df_s1["path"].dropna().unique().astype(int).tolist())}')
print(f'Platforms: {df_s1["platform"].value_counts().to_dict()}')
print(f'Orbit dirs: {df_s1["orbit_dir"].value_counts().to_dict()}')
print(f'Saved scene inventory')


In [ ]:
# 4. Recommended InSAR pairs - path 147 (Accra)
path147 = df_s1[df_s1['path'] == 147].sort_values('date').reset_index(drop=True)
print(f'Path 147 (ascending, covers Accra/Tema): {len(path147)} scenes')
print(f'  Range: {path147["date"].min().date()} to {path147["date"].max().date()}')

pairs = []
for i in range(len(path147) - 1):
    dt = (path147.loc[i+1,'date'] - path147.loc[i,'date']).days
    if dt <= 25:
        pairs.append({
            'master':                   path147.loc[i,'scene_name'],
            'slave':                    path147.loc[i+1,'scene_name'],
            'master_date':              str(path147.loc[i,'date'].date()),
            'slave_date':               str(path147.loc[i+1,'date'].date()),
            'temporal_baseline_days':   dt,
            'platform_master':          path147.loc[i,'platform'],
            'platform_slave':           path147.loc[i+1,'platform'],
        })

df_pairs = pd.DataFrame(pairs)
df_pairs.to_csv(TAB_DIR / 's1_insar_recommended_pairs_path147.csv', index=False)
print(f'Recommended pairs (<=25-day baseline): {len(df_pairs)}')
print(df_pairs.head(6)[['master_date','slave_date','temporal_baseline_days','platform_master']].to_string(index=False))


## Sentinel-1 InSAR Processing Workflow

### Option A: SNAP + MintPy (local)
```
1. Download S1 IW SLC pairs (see inventory above)
2. SNAP: TOPSAR Split → Apply Orbit → Back-Geocoding (ESD) →
         Interferogram → TOPSAR Deburst → Topo Phase Removal →
         Goldstein Filter → Export to MintPy
3. MintPy: mintpy.py inputs/accra.cfg
           PyAPS atmospheric correction → SNAPHU unwrapping →
           Velocity map → Displacement time series
```

### Option B: ASF HyP3 (recommended — cloud, no local SNAP needed)
```
pip install hyp3_sdk
python scripts/submit_hyp3_accra.py
```
Free for NASA EarthData users (up to 1,000 jobs/month).
Outputs: unwrapped interferogram, coherence, displacement GeoTIFF.

### Subsidence rate classification (CLAUDE.md §8.7.3)
| Category | Rate (mm/yr) |
|---|---|
| Stable | -2 to +2 |
| Low | -5 to -2 |
| Moderate | -10 to -5 |
| High | -20 to -10 |
| Very high | < -20 |


In [ ]:
# 5. Write HyP3 submission script
script_lines = [
    '#!/usr/bin/env python',
    '"""Submit S1 InSAR jobs to ASF HyP3 for Accra hotspot.',
    'pip install hyp3_sdk  then  python scripts/submit_hyp3_accra.py',
    '"""',
    'import hyp3_sdk as hyp3',
    'import pandas as pd',
    'from pathlib import Path',
    '',
    "pairs = pd.read_csv('outputs/tables/s1_insar_recommended_pairs_path147.csv')",
    'pairs = pairs.head(20)',
    '',
    'api = hyp3.HyP3()',
    'jobs = []',
    'for _, row in pairs.iterrows():',
    '    job = api.submit_insar_job(',
    "        granules=[row['master'], row['slave']],",
    "        name=f\"accra_{row['master_date']}_{row['slave_date']}\",",
    "        include_dem=True, include_displacement_maps=True, looks='20x4',",
    '    )',
    '    jobs.append(job)',
    "    print(f\"Submitted: {row['master_date']} to {row['slave_date']}\")",
    'print(f"Total jobs: {len(jobs)}")',
    'print("Check: https://hyp3.asf.alaska.edu")',
]

script_path = ROOT / 'scripts' / 'submit_hyp3_accra.py'
script_path.write_text('\n'.join(script_lines))
print('HyP3 script written to:', script_path)


In [ ]:
# 6. Proxy subsidence potential map
print('Building proxy subsidence potential ...')

trend_ds = xr.open_dataset(GW_DIR / 'gwsa_trend_pixelwise.nc')
gdsi_ds  = xr.open_dataset(GW_DIR / 'gdsi_drought_metrics_grid.nc')

slope_arr  = trend_ds['sens_slope_mm_yr'].values
drfreq_arr = gdsi_ds['drought_freq_pct'].values
lats = trend_ds['lat'].values
lons = trend_ds['lon'].values

# Normalise each component
def norm(x, invert=False):
    x = np.where(np.isnan(x), np.nanmedian(x), x)
    mn, mx = x.min(), x.max()
    n = (x - mn) / (mx - mn + 1e-9)
    return 1 - n if invert else n

# Declining GW trend (more negative slope = higher risk)
slope_norm  = norm(-slope_arr, invert=False)  # positive where GW declining
drfreq_norm = norm(drfreq_arr)

# DEM elevation (low = more susceptible)
dem_path = ROOT / 'data' / 'interim' / 'harmonized_grids' / 'dem_coastal50km_utm30n.tif'
if dem_path.exists():
    import rasterio
    with rasterio.open(dem_path) as src:
        dem_data = src.read(1).astype(float)
        nd = src.nodata
        dem_data[dem_data == nd] = np.nan
        transform = src.transform
    dem_coarse = np.full_like(slope_arr, np.nan)
    from rasterio.transform import rowcol
    for i, lat in enumerate(lats):
        for j, lon_v in enumerate(lons):
            try:
                row_c, col_c = rowcol(transform, lon_v, lat)
                r0 = max(0, row_c-50); r1 = min(dem_data.shape[0], row_c+50)
                c0 = max(0, col_c-50); c1 = min(dem_data.shape[1], col_c+50)
                patch = dem_data[r0:r1, c0:c1]
                valid = patch[~np.isnan(patch)]
                dem_coarse[i,j] = valid.mean() if len(valid) > 0 else np.nan
            except Exception:
                dem_coarse[i,j] = np.nan
    dem_coarse = np.clip(np.where(np.isnan(dem_coarse),
                                  np.nanmedian(dem_coarse), dem_coarse), 0, 100)
    elev_norm = 1.0 - norm(dem_coarse)
    has_dem = True
    print('  DEM layer included')
else:
    elev_norm = np.full_like(slope_arr, 0.5)
    has_dem = False
    print('  DEM not found, elevation set to neutral')

proxy = (slope_norm + drfreq_norm + elev_norm) / 3.0

proxy_da = xr.DataArray(
    proxy, dims=['lat', 'lon'],
    coords={'lat': lats, 'lon': lons},
    attrs={
        'long_name': 'Proxy subsidence potential (interim)',
        'note': 'PROXY ONLY: GWSA trend + drought frequency + elevation. '
                'Replace with InSAR velocity once HyP3 products are available.',
        'units': '0-1 normalised',
    }
)
xr.Dataset({'proxy_subsidence_potential': proxy_da}).to_netcdf(
    OUT_SUB / 'proxy_subsidence_potential_grid.nc')

trend_ds.close(); gdsi_ds.close()
print('Saved proxy subsidence potential')
print(f'Range: {proxy.min():.3f} - {proxy.max():.3f}')


In [ ]:
# 7. Classify proxy and save table
bins   = [0.0, 0.2, 0.4, 0.6, 0.8, 1.001]
labels = ['Very low','Low','Moderate','High','Very high']

records = []
for i, lat in enumerate(lats):
    for j, lon_v in enumerate(lons):
        cls = pd.cut([proxy[i,j]], bins=bins, labels=labels,
                     right=True, include_lowest=True)[0]
        records.append({'lat': lat, 'lon': lon_v,
                        'proxy_score': round(float(proxy[i,j]),4),
                        'proxy_class': str(cls)})
df_proxy = pd.DataFrame(records)
df_proxy.to_csv(OUT_SUB / 'proxy_subsidence_by_cell.csv', index=False)

dist = df_proxy['proxy_class'].value_counts().sort_index()
print('Proxy class distribution:')
for cls_val, cnt in dist.items():
    print(f'  {cls_val:12s}: {cnt}')
print('Saved:', OUT_SUB / 'proxy_subsidence_by_cell.csv')


In [ ]:
# 8. Figure: S1 scene availability
fig, axes = plt.subplots(2, 1, figsize=(13, 8))
fig.suptitle('Sentinel-1 IW SLC Availability — Coastal Ghana 2015-2024',
             fontsize=12, fontweight='bold')

df_s1['month'] = df_s1['date'].dt.to_period('M').dt.to_timestamp()
monthly = df_s1.groupby(['month','path']).size().unstack(fill_value=0)
path_colours = {1:'#1b7837', 45:'#762a83', 74:'#e08214', 118:'#2166ac', 147:'#d6604d'}
bottom = np.zeros(len(monthly))
for path_id in sorted(df_s1['path'].dropna().unique()):
    path_id = int(path_id)
    if path_id in monthly.columns:
        axes[0].bar(monthly.index, monthly[path_id].values, width=25,
                   bottom=bottom, color=path_colours.get(path_id,'grey'),
                   alpha=0.85, label=f'Path {path_id}')
        bottom += monthly[path_id].values
axes[0].axvline(pd.Timestamp('2021-12-23'), color='red', lw=1.5, ls='--',
               label='S1-B end of life')
axes[0].set_ylabel('Scenes/month'); axes[0].legend(fontsize=8, ncol=3)
axes[0].set_title('A  Scene count by orbital path', loc='left', fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

yearly = df_s1.groupby(['year','orbit_dir']).size().unstack(fill_value=0)
x = np.arange(len(yearly)); dirs = sorted(df_s1['orbit_dir'].dropna().unique())
dir_colours = {'ASCENDING':'#2171b5','DESCENDING':'#cb181d'}
for k, od in enumerate(dirs):
    if od in yearly.columns:
        axes[1].bar(x + k*0.4 - 0.2, yearly[od].values, width=0.4,
                   color=dir_colours.get(od,'grey'), alpha=0.85, label=od)
axes[1].set_xticks(x); axes[1].set_xticklabels(yearly.index.astype(str), rotation=45)
axes[1].set_ylabel('Scenes/year'); axes[1].legend(fontsize=9)
axes[1].set_title('B  Scene count by orbit direction', loc='left', fontsize=10)
axes[1].grid(axis='y', alpha=0.3)

fig.tight_layout(rect=[0,0,1,0.96])
fig.savefig(FIG_DIR / 's1_scene_availability.png', dpi=180, bbox_inches='tight')
plt.close(fig)
print('Saved:', FIG_DIR / 's1_scene_availability.png')


In [ ]:
# 9. Figure: proxy subsidence map
import cartopy.crs as ccrs
import cartopy.feature as cfeature

proxy_nc = xr.open_dataset(OUT_SUB / 'proxy_subsidence_potential_grid.nc')
proxy_g  = proxy_nc['proxy_subsidence_potential']

fig = plt.figure(figsize=(9, 9))
ax  = fig.add_subplot(111, projection=ccrs.PlateCarree())

cf = ax.pcolormesh(proxy_g.lon, proxy_g.lat, proxy_g.values,
                   cmap=plt.cm.YlOrRd, vmin=0, vmax=1,
                   transform=ccrs.PlateCarree())
ax.add_feature(cfeature.COASTLINE, lw=0.8)
ax.add_feature(cfeature.BORDERS, lw=0.5, linestyle=':')
ax.add_feature(cfeature.LAND, facecolor='#f5f5f5', zorder=0)

cbar = fig.colorbar(cf, ax=ax, shrink=0.65, pad=0.02)
cbar.set_label('Proxy subsidence potential (0-1)', fontsize=10)
ax.set_extent([-3.5, 1.5, 4.0, 11.5], crs=ccrs.PlateCarree())

hotspots = {'Accra':(-0.19,5.55),'Tema':(0.00,5.67),
            'Takoradi':(-1.75,4.90),'Cape Coast':(-1.25,5.12),'Keta':(0.99,5.92)}
for city,(lon_c,lat_c) in hotspots.items():
    ax.plot(lon_c, lat_c, 'k^', ms=7, transform=ccrs.PlateCarree(), zorder=5)
    ax.text(lon_c+0.05, lat_c+0.07, city, fontsize=8,
            transform=ccrs.PlateCarree(), zorder=5)

ax.set_title('Proxy Subsidence Potential — Coastal Ghana\n'
             '(PROXY: GWSA trend + drought freq + elevation; '
             'replace with InSAR velocity)', fontsize=10, fontstyle='italic')
gl = ax.gridlines(draw_labels=True, lw=0.3, alpha=0.5)
gl.top_labels = gl.right_labels = False

proxy_nc.close()
fig.tight_layout()
fig.savefig(FIG_DIR / 'proxy_subsidence_potential_map.png', dpi=180, bbox_inches='tight')
plt.close(fig)
print('Saved:', FIG_DIR / 'proxy_subsidence_potential_map.png')


In [ ]:
# 10. Stage 7 summary
print('=' * 60)
print('Stage 7 complete')
outputs_to_check = [
    OUT_INV / 's1_scene_inventory_ghana_2015_2024.csv',
    TAB_DIR / 's1_insar_recommended_pairs_path147.csv',
    OUT_SUB / 'proxy_subsidence_potential_grid.nc',
    OUT_SUB / 'proxy_subsidence_by_cell.csv',
    ROOT / 'scripts' / 'submit_hyp3_accra.py',
    FIG_DIR / 's1_scene_availability.png',
    FIG_DIR / 'proxy_subsidence_potential_map.png',
]
for p in outputs_to_check:
    tick = 'OK' if Path(p).exists() else 'MISSING'
    print(f'  [{tick}]  {Path(p).name}')

print()
print(f'S1 inventory: {len(df_s1)} scenes, paths {sorted(df_s1["path"].dropna().unique().astype(int).tolist())}')
print(f'Recommended pairs (path 147): {len(df_pairs)}')
print('Next: run scripts/submit_hyp3_accra.py OR process locally with SNAP+MintPy')
